# Drought quickstart — US Drought Monitor weekly polygons

Pull the most recent **US Drought Monitor (USDM)** weekly drought-class polygons through the `EarthLens`
facade. USDM releases one composite every **Thursday** UTC, valid the prior **Tuesday**, with five
features per release — one polygon per drought class **D0** (abnormally dry) through **D4**
(exceptional drought). The on-disk JSON / shapefile URL stem is keyed on the **Tuesday valid date**
(verified live; every Thursday URL returns 404).

The drought backend is a *mixed-output* backend: `OUTPUT_KIND` is set per instance from the resolved
catalog row. The USDM row is `vector`, so `download()` returns a
`pyramids.feature.collection.FeatureCollection` in EPSG:4326 — *not* a `list[Path]` like the SPEIbase /
EDO / GDO rasters.

## Setup

USDM is **anonymous HTTP** — no token, no extra dependency. The `EarthLens` facade plus `datetime` is
everything we need.

In [ ]:
import datetime as dt

from earthlens.core import EarthLens

from earthlens.drought import Catalog

## Pick the request window

USDM is weekly, so we want a window of at least seven days to land on at least one release. Any day
inside the window snaps to the **most recent already-released Tuesday composite** — the backend walks
back one extra week when the snapped Tuesday's release Thursday has not yet passed, so a query made on
the same Tuesday the composite is *valid for* gets the previous week's composite (this week's is not
live until Thursday).

We pin a deterministic window and a `today=` reference so the notebook re-executes to the same release
independently of when it is run.

In [ ]:
WINDOW_END = dt.date(2025, 6, 23)  # known-valid USDM week
WINDOW_START = WINDOW_END - dt.timedelta(days=14)
TODAY = dt.date(2025, 7, 1)  # well past WINDOW_END's release Thursday
print(
    'window:',
    WINDOW_START.isoformat(),
    'to',
    WINDOW_END.isoformat(),
    'today:',
    TODAY.isoformat(),
)

## Download the weekly polygons

The four drought facade keys (`drought` / `usdm` / `edo` / `gdo`) are discoverability aliases — all
resolve to the same backend and all require an explicit `dataset=` kwarg. We pass an explicit bbox over
the **continental United States** (USDM is national-extent by nature, so any sub-bbox simply clips the
polygons after the fetch). `today=` pins the release-window check so the snap result is deterministic.

In [ ]:
fc = EarthLens(
    data_source='usdm',
    start=WINDOW_START.isoformat(),
    end=WINDOW_END.isoformat(),
    variables=[],
    lat_lim=[24.0, 50.0],
    lon_lim=[-125.0, -66.0],
    dataset='usdm',
    today=TODAY,
).download(progress_bar=False)

print(type(fc).__name__, 'with', len(fc), 'features in EPSG:', fc.crs.to_epsg())

## Inspect the result

Every USDM feature carries:

* `OBJECTID` — provider-side row id.
* `DM` — drought class (`0` = D0 abnormally dry, … `4` = D4 exceptional drought).
* `Shape_Length` / `Shape_Area` — provider-side geometry stats in EPSG:4326 degrees.
* `release_date` — added by `earthlens` so a multi-week merge keeps the per-release attribution; the
  ISO date is the **Tuesday valid date**, not the Thursday release date.

Geometry is `MultiPolygon`.

In [ ]:
fc.head()

In [ ]:
summary = (
    fc[['release_date', 'DM', 'Shape_Area']]
    .groupby(['release_date', 'DM'])
    .agg(area=('Shape_Area', 'sum'), n=('DM', 'size'))
    .reset_index()
)
summary

## Quick map

A drought-class choropleth on a USA bbox — D0 (driest sliver) through D4 (extreme drought) in classic
USDM colours. We use the most recent snapped release so the map shows one consistent week.

In [ ]:
import matplotlib.pyplot as plt

USDM_COLORS = {0: '#ffff00', 1: '#fcd37f', 2: '#ffaa00', 3: '#e60000', 4: '#730000'}
latest_release = sorted(fc['release_date'].unique())[-1]
latest = fc[fc['release_date'] == latest_release].copy()
latest['color'] = latest['DM'].map(USDM_COLORS)

ax = latest.plot(
    color=latest['color'], edgecolor='#444', linewidth=0.4, figsize=(10, 5)
)
ax.set_title(f'USDM drought classes — release {latest_release}')
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')
plt.tight_layout()
plt.show()

## What the facade is doing under the hood

* `data_source='usdm'` resolves to the `earthlens.drought.Drought` backend; `dataset='usdm'` selects the
  USDM catalog row.
* The catalog row pins the transport (`usdm-geojson`) and the URL template (`{ymd}` substituted with
  the **Tuesday valid date**).
* Each snapped date triggers one HTTP fetch of the per-week JSON, parsed by `geopandas` into a
  `GeoDataFrame`, with the source CRS read from the payload's `crs` member (RFC 7946 default 4326).
* The frames are merged, reprojected to EPSG:4326 when needed, clipped to the requested bbox, and
  wrapped in a `FeatureCollection`.
* On success the backend logs the per-source attribution (`USDM: U.S. Drought Monitor — public-domain
  weekly composite produced by NDMC / UNL / USDA / NOAA. Cite the National Drought Mitigation Center.`).

In [ ]:
Catalog().get('usdm')

## Where to go next

* The catalog ships 39 datasets total — **USDM (1)**, **EDO (15)**, **GDO (17)**, **SPEIbase (6)** —
  see the `catalog_explorer.ipynb` notebook for the full list.
* SPEIbase (raster monthly NetCDF) is wired and returns a `list[Path]` of per-month GeoTIFFs.
  Raster transports **require** an explicit `path=` to avoid silently writing hundreds of MB into the
  current working directory.
* EDO / GDO indicators (Copernicus OGC WCS) are wired in the catalog and routed by the backend, but
  the fetch raises `NotImplementedError` until the pyramids temporal `read_wcs` extension (the
  cross-repo `PY-A` task) ships.
* Vector backends like USDM **reject** `aggregate=` — drought-class polygons have no gridded reduction.